In [1]:
! pip install cv2
! pip install numpy
! pip install matplotlib
! pip install skimage
! pip install scipy

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)
ERROR: No matching distribution found for cv2
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
from scipy.spatial.distance import euclidean



In [3]:
# =======================
# EXPERIMENT PARAMETERS
# =======================

video_path = "/content/myelin 1-10102025144351-0000.avi"

laser_power_W = 1.0      # 1000 mW
pixel_size_um = 0.01    # microscope calibration (µm / pixel)

n_medium = 1.33          # water
c = 3e8                  # speed of light (m/s)
Q = 0.1                  # trapping efficiency (0.05–0.2 typical)

min_edge_points = 100    # reject noisy frames


In [4]:
def optical_force(P, n=1.33, Q=0.1):
    return n * P * Q / c


In [5]:
def skeleton_length(binary_img):
    skel = skeletonize(binary_img > 0)
    points = np.column_stack(np.where(skel))
    if len(points) < 2:
        return None

    length = 0
    for i in range(len(points) - 1):
        length += euclidean(points[i], points[i + 1])

    return length


In [6]:
cap = cv2.VideoCapture(video_path)

radii_px = []
lengths_px = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Edge detection
    edges = cv2.Canny(gray, 50, 150)

    points = np.column_stack(np.where(edges > 0))
    if len(points) < min_edge_points:
        continue

    # =====================
    # CURVATURE (R)
    # =====================
    (yc, xc), radius = cv2.minEnclosingCircle(points)
    radii_px.append(radius)

    # =====================
    # LENGTH (L)
    # =====================
    _, binary = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    L_px = skeleton_length(binary)
    if L_px is not None:
        lengths_px.append(L_px)

cap.release()


In [7]:
R_um = np.mean(radii_px) * pixel_size_um
L_um = np.mean(lengths_px) * pixel_size_um

R_m = R_um * 1e-6
L_m = L_um * 1e-6


In [11]:
F = optical_force(laser_power_W, n_medium, Q)

k_c = F * R_m**2 / (2 * np.pi * L_m)


In [12]:
R_std = np.std(np.array(radii_px) * pixel_size_um) * 1e-6
L_std = np.std(np.array(lengths_px) * pixel_size_um) * 1e-6

k_c_err = k_c * np.sqrt(
    (2 * R_std / R_m)**2 +
    (L_std / L_m)**2
)


In [13]:
print("=================================")
print("Elastic Modulus from Optical Trap")
print("=================================")
print(f"Laser Power: {laser_power_W*1000:.0f} mW")
print(f"Radius R: {R_um:.2f} µm")
print(f"Length L: {L_um:.2f} µm")
print(f"Force F: {F:.2e} N")
print(f"Bending modulus k_c: {k_c:.2e} ± {k_c_err:.2e} J")


Elastic Modulus from Optical Trap
Laser Power: 1000 mW
Radius R: 3.41 µm
Length L: 2017.97 µm
Force F: 4.43e-10 N
Bending modulus k_c: 4.06e-19 ± 2.73e-19 J
